# **Demo 01: Building a CrewAI-Powered Agent With Claude**


**Objective:** To demonstrate how to build a multi-agent system using CrewAI that intelligently identifies queries requiring real-time information and retrieves the most up-to-date data accordingly.

**Prerequisites:** Anthropic API key, Tavily key

**Tools required:** Python (Google Colab)

**Scenario:** A product manager at a digital knowledge platform needs to manage an increasing volume of user queries, many of which require real-time updates and dynamic information. At present, these queries are reviewed and answered manually, leading to delays, inconsistencies, and rising operational costs. The goal is to automate the query-handling process using a multi-agent system that can identify when real-time data retrieval is needed and provide users with fast, accurate, and context-aware responses while maintaining efficiency and cost control.

In [ ]:
# Step 1: Install the required libraries
!pip install -q "crewai[anthropic]" langchain langchain-community langchain-tavily tavily-python pydantic litellm nest_asyncio

In [ ]:
import os
import requests
import nest_asyncio
from crewai.llm import LLM
from crewai import Agent, Task, Crew
from crewai.tools import BaseTool

# Colab's event loop needs this patch for CrewAI's async kickoff
nest_asyncio.apply()

In [ ]:
# Step 2: Load API keys from Colab Secrets (the key icon in the left sidebar)
# Add ANTHROPIC_API_KEY and TAVILY_API_KEY there before running this cell
from google.colab import userdata

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")

In [ ]:
# ---- Custom CrewAI Tool for Web Search ----
class TavilySearchTool(BaseTool):
    name: str = "Web Search"
    description: str = "Search the web for recent information."

    def _run(self, query: str):
        url = "https://api.tavily.com/search"

        payload = {
            "api_key": os.environ["TAVILY_API_KEY"],
            "query": query,
            "max_results": 3
        }

        response = requests.post(url, json=payload, timeout=30)
        response.raise_for_status()
        data = response.json()

        results = []
        for r in data.get("results", []):
            results.append(f"{r['title']} - {r['url']}")

        return "\n".join(results) if results else "No web results found."


search_tool = TavilySearchTool()

# ---- Claude LLM via CrewAI's native LiteLLM bridge ----
# Swap the model string for whichever Claude model your API access supports
llm = LLM(
    model="anthropic/claude-sonnet-4-5-20250929",
    api_key=os.environ["ANTHROPIC_API_KEY"],
    max_tokens=3000,
    is_litellm=True
)

In [ ]:
# Researcher agent
researcher = Agent(
    role="AI Researcher",
    goal="Find the latest advancements in AI for FMCG",
    backstory="You are an expert in artificial intelligence and stay updated with the latest research trends in FMCG.",
    verbose=True,
    allow_delegation=False,
    llm=llm,
    max_iter=3,
    tools=[search_tool]
)

# Writer agent
writer = Agent(
    role="Technical Writer",
    goal="Summarize research into an executive report",
    backstory="You are an experienced technical writer with expertise in summarizing research for executives.",
    verbose=True,
    allow_delegation=False,
    llm=llm,
    tools=[search_tool]
)

In [ ]:
# ---- Tasks ----
task_research = Task(
    description="Search the web and identify the top 3 recent advancements in AI for FMCG.",
    expected_output="Detailed notes explaining three recent AI advancements in FMCG with examples.",
    agent=researcher
)

task_write = Task(
    description="""
Write a concise executive summary using the research notes.

Requirements:
- Maximum 100 words
- Use bullet points
- Focus only on the 3 key advancements
""",
    expected_output="Executive summary of AI advancements in FMCG.",
    agent=writer,
    context=[task_research]
)

In [ ]:
# ---- Crew ----
crew = Crew(
    agents=[researcher, writer],
    tasks=[task_research, task_write],
    verbose=True
)

result = await crew.kickoff_async()

print("\nFinal Output:\n")
print(result.raw)